In [1]:
"""
Financial Analysis: AI Finance Tracker
Analyzes personal finance transaction data to surface spending trends,
category breakdowns, savings behavior, and anomalies.
"""

'\nFinancial Analysis: AI Finance Tracker\nAnalyzes personal finance transaction data to surface spending trends,\ncategory breakdowns, savings behavior, and anomalies.\n'

In [2]:
from pathlib import Path

In [3]:
import pandas as pd
import numpy as np
import matplotlib

In [4]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [6]:
if "__file__" in globals():
    base_candidates = [Path(__file__).resolve().parent.parent]
else:
    cwd = Path.cwd()
    base_candidates = [cwd, cwd / "analytics", cwd.parent, cwd.parent / "analytics"]

In [7]:
BASE_DIR = None
for candidate in base_candidates:
    if (candidate / "data" / "sample_transactions.csv").exists():
        BASE_DIR = candidate
        break

In [8]:
if BASE_DIR is None:
    BASE_DIR = Path.cwd()

In [9]:
DATA_PATH = BASE_DIR / "data" / "sample_transactions.csv"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
# ── Load data ─────────────────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df["month"] = df["date"].dt.to_period("M")

In [11]:
print(f"Loaded {len(df)} transactions from {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

Loaded 253 transactions from 2025-07-01 to 2026-06-20


,transaction_id,date,type,category,amount,description,month
0,TXN-00001,2025-07-01,income,Salary,51394.27,Monthly salary,2025-07
1,TXN-00040,2025-07-01,expense,Investments,3645.16,Mutual fund SIP,2025-07
2,TXN-00041,2025-07-01,expense,Subscriptions,914.71,Membership fee,2025-07
3,TXN-00023,2025-07-01,expense,Rent,12180.00,Apartment rent,2025-07
4,TXN-00022,2025-07-01,income,Salary,47904.09,Payroll deposit,2025-07


In [12]:
# ── 1. Monthly income vs expense trend ───────────────
monthly = df.groupby(["month", "type"])["amount"].sum().unstack(fill_value=0)
monthly.index = monthly.index.astype(str)

In [13]:
ax = monthly.plot(kind="line", marker="o", figsize=(10, 5))
ax.set_title("Monthly Income vs Expense Trend")
ax.set_ylabel("Amount (₹)")
ax.set_xlabel("Month")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "monthly_trend.png", dpi=120)
plt.show()

C:\Users\home\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:8: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  


In [14]:
avg_income = monthly["income"].mean()
avg_expense = monthly["expense"].mean()
print(f"Insight: Average monthly income ₹{avg_income:,.0f} vs average expense ₹{avg_expense:,.0f} "
      f"→ average monthly surplus of ₹{avg_income - avg_expense:,.0f}")

Insight: Average monthly income ₹62,955 vs average expense ₹78,914 → average monthly surplus of ₹-15,959


In [15]:
# ── 2. Category-wise spending breakdown ──────────────
category_spend = df[df["type"] == "expense"].groupby("category")["amount"].sum().sort_values(ascending=False)
category_pct = (category_spend / category_spend.sum() * 100).round(1)

In [16]:
ax = category_spend.plot(kind="bar", color=sns.color_palette("Blues_r", len(category_spend)))
ax.set_title("Total Spending by Category")
ax.set_ylabel("Amount (₹)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "category_breakdown.png", dpi=120)
plt.show()

C:\Users\home\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  import sys


In [17]:
print("Top 3 spending categories:")
print(category_pct.head(3))

Top 3 spending categories:
category
Groceries    31.5
Rent         16.5
Transport    16.1
Name: amount, dtype: float64


In [18]:
# ── 3. Savings rate over time ────────────────────────
monthly["savings"] = monthly["income"] - monthly["expense"]
monthly["savings_rate_%"] = (monthly["savings"] / monthly["income"] * 100).round(1)

In [19]:
ax = monthly["savings_rate_%"].plot(kind="line", marker="o", color="green")
ax.set_title("Savings Rate Over Time")
ax.set_ylabel("Savings Rate (%)")
ax.axhline(20, color="red", linestyle="--", label="Recommended 20% threshold")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "savings_rate.png", dpi=120)
plt.show()

C:\Users\home\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:9: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  if __name__ == "__main__":


In [20]:
print(f"Insight: Average savings rate is {monthly['savings_rate_%'].mean():.1f}%. "
      f"Months below 20%: {(monthly['savings_rate_%'] < 20).sum()} of {len(monthly)}")

Insight: Average savings rate is -44.8%. Months below 20%: 12 of 12


In [21]:
# ── 4. Month-over-month expense growth rate ──────────
monthly["expense_growth_%"] = monthly["expense"].pct_change().round(3) * 100

In [22]:
ax = monthly["expense_growth_%"].plot(kind="bar", color=monthly["expense_growth_%"].apply(lambda x: "red" if x > 0 else "green"))
ax.set_title("Month-over-Month Expense Growth Rate")
ax.set_ylabel("Growth (%)")
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "expense_growth.png", dpi=120)
plt.show()

C:\Users\home\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:8: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  


In [23]:
# ── 5. Anomaly / outlier detection (IQR method) ──────
expenses = df[df["type"] == "expense"].copy()
Q1 = expenses["amount"].quantile(0.25)
Q3 = expenses["amount"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

In [24]:
outliers = expenses[expenses["amount"] > upper_bound].sort_values("amount", ascending=False)

In [25]:
print(f"Outlier threshold (IQR method): ₹{upper_bound:,.0f}")
print(f"Found {len(outliers)} anomalous transactions:")
print(outliers[["date", "category", "amount", "description"]].to_string(index=False))

Outlier threshold (IQR method): ₹10,075
Found 19 anomalous transactions:
      date     category    amount             description
2026-02-10    Transport  21071.63              Ride share
2025-07-01       Dining  18056.44           Food delivery
2026-05-04    Groceries  15708.41        Grocery delivery
2026-05-01         Rent  14135.39          Apartment rent
2026-04-01         Rent  13926.49         Housing payment
2026-03-01         Rent  13720.68         Housing payment
2026-02-01         Rent  13517.91          Apartment rent
2026-01-01         Rent  13318.14         Housing payment
2025-12-01         Rent  13121.32            Monthly rent
2025-09-06    Transport  13043.41              Metro pass
2025-11-01         Rent  12927.41         Housing payment
2025-10-01         Rent  12736.36         Housing payment
2025-09-01         Rent  12548.14            Monthly rent
2025-08-01         Rent  12362.70            Monthly rent
2025-07-01         Rent  12180.00          Apartment rent

In [26]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=expenses, x="category", y="amount")
plt.xticks(rotation=45, ha="right")
plt.title("Expense Distribution by Category (Outlier Detection)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "outlier_detection.png", dpi=120)
plt.show()

C:\Users\home\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  import sys


In [27]:
# ── 6. Budget vs actual variance (assumed budgets) ───
assumed_budgets = {
    "Rent": 12000, "Groceries": 6000, "Dining": 2500, "Transport": 2500,
    "Utilities": 2500, "Entertainment": 1500, "Shopping": 3000,
    "Healthcare": 2000, "Investments": 6000, "Subscriptions": 800,
}

In [28]:
latest_month = df["month"].max()
latest_spend = df[(df["type"] == "expense") & (df["month"] == latest_month)].groupby("category")["amount"].sum()

In [29]:
variance_df = pd.DataFrame({
    "Budget": pd.Series(assumed_budgets),
    "Actual": latest_spend
}).fillna(0)
variance_df["Variance"] = variance_df["Actual"] - variance_df["Budget"]
variance_df["Variance_%"] = (variance_df["Variance"] / variance_df["Budget"] * 100).round(1)
variance_df = variance_df.sort_values("Variance_%", ascending=False)

In [30]:
print(f"\nBudget vs Actual — {latest_month}:")
print(variance_df.to_string())


Budget vs Actual — 2026-06:
               Budget    Actual  Variance  Variance_%
Dining           2500  11539.89   9039.89       361.6
Groceries        6000  18137.92  12137.92       202.3
Transport        2500   7490.86   4990.86       199.6
Shopping         3000   8057.28   5057.28       168.6
Healthcare       2000   4899.36   2899.36       145.0
Entertainment    1500   2984.82   1484.82        99.0
Investments      6000      0.00  -6000.00      -100.0
Rent            12000      0.00 -12000.00      -100.0
Subscriptions     800      0.00   -800.00      -100.0
Utilities        2500      0.00  -2500.00      -100.0


In [31]:
over_budget = variance_df[variance_df["Variance"] > 0]
print(f"\nInsight: {len(over_budget)} categories exceeded budget in {latest_month}, "
      f"led by {over_budget.index[0]} at {over_budget['Variance_%'].iloc[0]:.0f}% over.")


Insight: 6 categories exceeded budget in 2026-06, led by Dining at 362% over.


In [32]:
print("\n✅ Analysis complete. Charts saved to analytics/outputs/")


✅ Analysis complete. Charts saved to analytics/outputs/
